# Workflow Optimizer

Describe a task; an agent designs candidate **workflow programs**, iterates to make
them cheaper without losing accuracy, and you pick the point you want on the
accuracy/cost **Pareto frontier**.

A workflow is just a Python function:

```python
def solve(question, call_model):
    ...            # any strategy: one call, chain-of-thought, vote, route, decompose
    return answer
```

`call_model` is the single metered gateway to a model — which is how cost is measured
and capped no matter what the program does.

This notebook is a thin driver over the `workflow_optimizer` package; the code lives in `src/workflow_optimizer/`
and every knob is in `config/`. **Every cell makes real API calls — set `ANTHROPIC_API_KEY`
first.**

## Configure

Pick a task file from `config/task/` and override any config key inline. See
`config/config.yaml` for the full set.

In [ ]:
from workflow_optimizer import Session, analysis, report
from workflow_optimizer.optimizer import optimize

session = Session.load("clinical_notes", ["designer.rounds=2"])
cfg = session.cfg

print(cfg.task.seed_prompt)
print("models:", ", ".join(session.catalog.ids))

## Step 1 · Analyze the task

One structured call infers the task description, the grading check (numeric /
exact-match / LLM-judge), and — for free-form tasks — a judge rubric, which is then
calibration-checked against example answers. It also generates a labeled dataset if
the task didn't supply one, split into **dev** (the designer may tune on this) and a
held-out **test** set.

In [ ]:
benchmark = analysis.build_benchmark(cfg, session.client)
benchmark.analysis

## Steps 2-3 · Design and optimize

When `designer.research` is set, a web-research phase runs first: the agent searches for
what works on this task and writes research_notes.md, handed to every design round.
The design agent then runs once per round in a subprocess, driven by the skills in `skills/`.
Round 1 designs a diverse initial set; each later round sees the whole archive — every
workflow's code, its dev accuracy and cost, and the dev examples it got wrong — and is
asked for new ones that extend the accuracy/cost frontier. Every candidate is scored on
dev; the dev frontier is then re-scored on the held-out test split.

> This is the expensive part — it runs the agent `designer.rounds` times and scores every
> candidate. Lower `designer.rounds` to spend less.

In [ ]:
search = optimize(cfg, benchmark, session.evaluator(benchmark.grader))

## Step 4 · The tradeoffs

The frontier, the two constrained picks (best under a budget, cheapest above an accuracy
floor), and a plot of every finalist.

In [ ]:
report.summarize(search, cfg)
report.plot(search, show=True);

## Step 5 · Choose your methodology

Every workflow below is a genuine option at a different accuracy/cost point. Read the
code, then pick one — `chosen.code` is what you'd ship.

In [ ]:
report.print_code(search)

from workflow_optimizer import TEST
from workflow_optimizer.pareto import pareto_front

CHOICE = pareto_front(search.finalists, on=TEST)[-1].name      # default: most accurate on the frontier
chosen = next(c for c in search.finalists if c.name == CHOICE)

print(f"CHOSEN: {chosen.name}  (accuracy {chosen.test.accuracy:.2f}, ${chosen.test.cost:.5f}/query)\n")
print(chosen.code.strip())
print("\nsaved:", report.save(search, cfg))